# Reproduce MisakaNet's retrieval benchmark in five minutes

This notebook runs the repository's own retrieval benchmark against the **live** endpoint — the
same 42 queries the CI gate uses (20 English, 22 CJK), plus the corpus sizes that explain the
difference between the gate and production.

It is deliberately small: `scripts/bench_production_recall.py` has **no dependencies beyond the
Python standard library**, so the only thing to install is the repository itself.

> The script is not a gate and its exit code is a signal for a human: `1` means the live service
> scored below the gate's floor, which is either the corpus/projection difference printed above
> it or a real regression. Read the table, not the colour.

Open it directly in Colab:

```
https://colab.research.google.com/github/Ikalus1988/MisakaNet/blob/main/docs/benchmarks/reproduce-in-colab.ipynb
```

## 1. Clone the repository (the queries and the local corpus live in it)

In [ ]:
!git clone --depth 1 https://github.com/Ikalus1988/MisakaNet.git
%cd MisakaNet

## 2. The comparison: the gate's 42 queries against production

Pacing: the anonymous read path has a per-address burst guard, so the default 1.5 s between
queries is not padding — `429` responses are retried with backoff, not counted as misses.
`--top 3` is the gate's rank window.

In [ ]:
!python3 scripts/bench_production_recall.py --top 3

## 3. The local path, for contrast

The local stdio server searches this checkout's `data/lessons.json` with BM25 and no network.
The numbers above are the *hosted* path over D1's rich projection; they are allowed to differ,
and the script prints both corpus sizes so the difference is not mistaken for a regression.

In [ ]:
!python3 search_knowledge.py "ReadTimeoutError" --lessons --remote
!python3 search_knowledge.py "ReadTimeoutError" --lessons

## 4. Where the published numbers come from

* gate: `workers/search-dual-floor.test.mjs` (runs in CI, over `data/lessons.json`)
* live comparison: `scripts/bench_production_recall.py` (this notebook)
* the 267-query report: [`docs/benchmarks/retrieval-l1-267-2026-10-27.md`](https://github.com/Ikalus1988/MisakaNet/blob/main/docs/benchmarks/retrieval-l1-267-2026-10-27.md)
* the dataset the report used: <https://data.misakanet.org/latest.json>

If a row moves, the first question is always **which corpus** — not which code.